**Status for the second revision (October 2026).** The scripts in `revision_pipeline/` are canonical. This notebook documents the corpus extraction and has no stored outputs. The paper's candidate pool is not re-extracted: the 1,824 raw candidates of the original submission are committed as `revision_pipeline/raw_candidates_1824.txt`, `revision_pipeline/normalize.py` reduces them to 1,652 lemmas, and `revision_pipeline/merge_core.py` removes break and practice, which the merge with Stanny (2016) added to the core, giving the 1,650 candidates in `revision_pipeline/candidates_final.txt`. Rerunning the extraction cells writes new lists to `results/` and does not change those files.

In [ ]:
'''
BT 2 - Data collection.ipynb
Author: Muhammad Talha & Jingchuan Shi
Supervisor: Dr. Ahmed Qureshi
Created 2019/9/6, last modified 2026/10/28 at University of Alberta.
Released under the MIT License (see LICENSE).
'''

# === BT-2: alternative spaCy + WordNet extractor (not used; the collection cells below use Stanza, Section 2.2) ===
from pathlib import Path
import spacy
from nltk.corpus import wordnet as wn

try:
    nlp = spacy.load("en_core_web_lg")
except OSError:
    raise RuntimeError("Install model via: python -m spacy download en_core_web_lg")

nlp.max_length = 3_000_000

def is_valid_verb(w: str) -> bool:
    return any(ss.pos() == "v" for ss in wn.synsets(w))

def extract_verbs(text: str):
    doc = nlp(text)
    out=[]
    for tok in doc:
        if tok.pos_ in {"VERB","AUX"}:
            lem = tok.lemma_.lower().strip()
            if lem.isalpha() and is_valid_verb(lem):
                out.append(lem)
    return out


## Corpus location

The extraction corpus (the PDF sources) is included in this repository in the folder `Papers & Books Used for Extraction`. These are third-party works; their copyright status and the terms under which they are included are set out in `Papers & Books Used for Extraction/NOTICE.md`. The list of processed sources is `results/BTdone_paper_list.txt`. To run the cells below on another copy of the corpus, set the environment variable `BT_CORPUS_DIR` to that folder; if it is not set, the notebook uses `Papers & Books Used for Extraction` in the repository root.

In [ ]:
# ===================== BT-2: Collect new verbs =====================
import os
from pathlib import Path
import stanza
from nltk.corpus import wordnet as wn

# 0) NLP pipeline (reuse downloaded resources when available)
try:
    from stanza.pipeline.core import DownloadMethod
    nlp = stanza.Pipeline(
        lang="en",
        processors="tokenize,pos,lemma",
        tokenize_pretokenized=False,
        download_method=DownloadMethod.REUSE_RESOURCES
    )
except Exception:
    nlp = stanza.Pipeline(lang="en", processors="tokenize,pos,lemma", tokenize_pretokenized=False)

# 1) Root and standard folders
BASE_DIR   = Path.cwd()
RESULTS_DIR = BASE_DIR / "results"
LOGS_DIR    = BASE_DIR / "logs"
CORPUS_DIR  = Path(os.environ.get("BT_CORPUS_DIR", BASE_DIR / "Papers & Books Used for Extraction"))
for d in (RESULTS_DIR, LOGS_DIR, CORPUS_DIR):
    d.mkdir(parents=True, exist_ok=True)

core_source_path = RESULTS_DIR / "BTverblist_core.txt"    # from BT-1
new_source_path  = RESULTS_DIR / "BTverblist_new.txt"     # grows here
done_source_path = RESULTS_DIR / "BTdone_paper_list.txt"  # processed files
progress_path    = LOGS_DIR    / "BTgain_progress.txt"    # progress log
for p in (core_source_path, new_source_path, done_source_path, progress_path):
    p.touch(exist_ok=True)

# 2) Helpers
def extract_text(file_path: Path) -> str:
    """Read text from .pdf (via Tika) or .txt (utf-8). Return '' if not available."""
    if file_path.suffix.lower() == ".pdf":
        try:
            from tika import parser as tika_parser
            raw = tika_parser.from_file(str(file_path))
            txt = (raw or {}).get("content") or ""
            return txt
        except Exception:
            return ""
    else:  # .txt
        try:
            return file_path.read_text(encoding="utf-8", errors="ignore")
        except Exception:
            return ""

def as_abs_str(p: Path) -> str:
    try:
        return str(p.resolve())
    except Exception:
        return str(p.absolute())

# 3) Load existing lists
core_verbs = [l.strip() for l in core_source_path.read_text(encoding="utf-8").splitlines() if l.strip()]
verbs      = [l.strip() for l in new_source_path.read_text(encoding="utf-8").splitlines() if l.strip()]
old_size   = len(verbs)
print(f"\nCurrent data size: {old_size}")

done_papers = {line.strip() for line in done_source_path.read_text(encoding="utf-8").splitlines() if line.strip()}
done_abs = {as_abs_str(Path(s)) for s in done_papers}

# 4) Files to process (recursive scan of the corpus directory)
files = []
for p in CORPUS_DIR.rglob("*"):
    if p.is_file() and p.suffix.lower() in {".pdf", ".txt"} and as_abs_str(p) not in done_abs:
        files.append(p)

core_set  = set(core_verbs)
verbs_set = set(verbs)

# 5) Legacy verb criterion
#    - Use XPOS only: {VB, VBD, VBG, VBN, VBP, VBZ}
#    - Lemma lowercased; no extra normalization/regex
VB_TAGS = {"VB","VBD","VBG","VBN","VBP","VBZ"}

for file_path in files:
    raw_text = extract_text(file_path)
    if not raw_text:
        with done_source_path.open("a", encoding="utf-8", newline="\n") as n:
            n.write(as_abs_str(file_path) + "\n")
        done_abs.add(as_abs_str(file_path))
        continue

    # basic PDF hyphenation/newline cleanup
    raw_text = raw_text.replace("-\n", "").replace("\n", " ")

    # NLP pass
    try:
        doc = nlp(raw_text)
    except Exception:
        with done_source_path.open("a", encoding="utf-8", newline="\n") as n:
            n.write(as_abs_str(file_path) + "\n")
        done_abs.add(as_abs_str(file_path))
        continue

    new_verbs = []
    for sent in doc.sentences:
        for w in sent.words:
            if w.xpos in VB_TAGS:
                lemma = (w.lemma or "").lower()
                if lemma and lemma not in core_set and lemma not in verbs_set and lemma not in new_verbs:
                    new_verbs.append(lemma)
                    verbs.append(lemma)
                    verbs_set.add(lemma)

    # Append new verbs
    if new_verbs:
        with new_source_path.open("a", encoding="utf-8", newline="\n") as m:
            for w in new_verbs:
                m.write(w + "\n")

    # Mark file as processed (absolute-path tracking)
    with done_source_path.open("a", encoding="utf-8", newline="\n") as n:
        n.write(as_abs_str(file_path) + "\n")
    done_abs.add(as_abs_str(file_path))

    # Progress report and log
    gain = len(verbs) - old_size
    print(f"Loaded {file_path.name}. Total verbs: {len(verbs)} (+{gain}).")
    old_size = len(verbs)
    with progress_path.open("a", encoding="utf-8", newline="\n") as n:
        n.write(f"{len(verbs)} {gain}\n")

# 6) WordNet validation
valid_verbs = [v for v in verbs if wn.synsets(v, pos="v")]
print("# of verbs identified by WordNet:", len(valid_verbs))


In [ ]:
# ===== BT-2: Rebuild list from corpus (force reprocess, legacy criterion) =====
import os
from pathlib import Path
import stanza
from nltk.corpus import wordnet as wn

# 0) Setup
BASE_DIR   = Path.cwd()
RESULTS_DIR = BASE_DIR / "results"
LOGS_DIR    = BASE_DIR / "logs"
CORPUS_DIR  = Path(os.environ.get("BT_CORPUS_DIR", BASE_DIR / "Papers & Books Used for Extraction"))
for d in (RESULTS_DIR, LOGS_DIR, CORPUS_DIR):
    d.mkdir(parents=True, exist_ok=True)

core_source_path = RESULTS_DIR / "BTverblist_core.txt"
new_source_path  = RESULTS_DIR / "BTverblist_new.txt"
done_source_path = RESULTS_DIR / "BTdone_paper_list.txt"
progress_path    = LOGS_DIR    / "BTgain_progress.txt"
for p in (core_source_path, new_source_path, done_source_path, progress_path):
    p.touch(exist_ok=True)

# 1) NLP
try:
    from stanza.pipeline.core import DownloadMethod
    nlp = stanza.Pipeline(lang="en", processors="tokenize,pos,lemma",
                          tokenize_pretokenized=False,
                          download_method=DownloadMethod.REUSE_RESOURCES)
except Exception:
    nlp = stanza.Pipeline(lang="en", processors="tokenize,pos,lemma", tokenize_pretokenized=False)

# 2) Load lists
core_verbs = [l.strip() for l in core_source_path.read_text(encoding="utf-8").splitlines() if l.strip()]
verbs      = [l.strip() for l in new_source_path.read_text(encoding="utf-8").splitlines() if l.strip()]
old_size   = len(verbs)
print(f"\nCurrent data size: {old_size}")

# 3) Build file list (IGNORE done list for this run)
files = [p for p in CORPUS_DIR.iterdir()
         if p.is_file() and p.suffix.lower() in {".pdf", ".txt"}]

# 4) Readers
def extract_text(p: Path) -> str:
    if p.suffix.lower() == ".pdf":
        try:
            from tika import parser as tika_parser
            raw = tika_parser.from_file(str(p))
            return (raw or {}).get("content") or ""
        except Exception:
            return ""
    try:
        return p.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        return ""

# 5) Legacy verb criterion (XPOS VB* only)
VB_TAGS = {"VB","VBD","VBG","VBN","VBP","VBZ"}
core_set  = set(core_verbs)
verbs_set = set(verbs)

for fp in files:
    txt = extract_text(fp)
    if not txt:
        print(f"(skip unreadable) {fp.name}")
        continue
    txt = txt.replace("-\n","").replace("\n"," ")

    try:
        doc = nlp(txt)
    except Exception:
        print(f"(skip nlp error) {fp.name}")
        continue

    new_verbs = []
    for sent in doc.sentences:
        for w in sent.words:
            if w.xpos in VB_TAGS:
                lemma = (w.lemma or "").lower()
                if lemma and lemma not in core_set and lemma not in verbs_set and lemma not in new_verbs:
                    new_verbs.append(lemma)
                    verbs.append(lemma)
                    verbs_set.add(lemma)

    if new_verbs:
        with new_source_path.open("a", encoding="utf-8", newline="\n") as m:
            for w in new_verbs:
                m.write(w + "\n")

    # The done list is not updated here, so files are reprocessed on every run

    gain = len(verbs) - old_size
    print(f"Processed {fp.name}. Total verbs: {len(verbs)} (+{gain}).")
    old_size = len(verbs)
    with progress_path.open("a", encoding="utf-8", newline="\n") as n:
        n.write(f"{len(verbs)} {gain}\n")

# 6) WordNet validation
wn_verbs = [v for v in verbs if wn.synsets(v, pos="v")]
print("# of verbs identified by WordNet:", len(wn_verbs))



## Candidate normalization

The raw candidate pool produced by the collection pass above still contained noise: inflected surface forms that survived tagging, British spelling variants of verbs already present in the core list or elsewhere in the pool, pure auxiliaries (be, have, do), and candidates whose lemma is itself a core Bloom verb (leakage, because the exclusion filter compares surface forms rather than lemmas). The cell below cleans the pool with an explicit audit trail, applying these rules in order:

1. WordNet verb lemmatization (surface form to lemma, lowercased).
2. British to American spelling merges, applied only when the American form already exists in the core list or in the candidate pool, so that a merge always resolves a real duplicate and never invents a new form.
3. Drop the pure auxiliaries be, have, do.
4. Drop candidates whose normalized lemma is a core verb.
5. Deduplicate: candidates that collapse to the same lemma keep a single entry.

On the pool used in the paper this reduces 1824 raw candidates to 1652 cleaned lemmas: 69 core-verb leaks removed, 98 duplicates merged, and 5 auxiliaries dropped. The cell below runs `revision_pipeline/normalize.py`, the canonical implementation, which reads the committed raw pool `revision_pipeline/raw_candidates_1824.txt` and the 358-verb core the normalization was run against (`revision_pipeline/core_notebook_original.csv`). Outputs are `results/cleaned_candidates.txt` and the per-candidate audit trail `results/normalization_audit.csv`; they are identical to the committed copies in `revision_pipeline/`. `revision_pipeline/merge_core.py` then removes break and practice, which the merge with Stanny (2016) added to the core, giving the 1,650 candidates in `revision_pipeline/candidates_final.txt`.

In [ ]:
# ===== BT-2: Candidate normalization (lemmatize, merge spellings, drop noise) =====
# Canonical implementation: revision_pipeline/normalize.py (reads the committed raw pool
# revision_pipeline/raw_candidates_1824.txt, so it runs on a fresh clone).
# --outdir results keeps the committed copies in revision_pipeline/ untouched.
%run "revision_pipeline/normalize.py" --outdir results